### Learning Objectives
  - Master all 6 types of joins
  - Understand join strategies (Broadcast, Sort-Merge,Shuffle)
  - Know when to use each join
  - Handle Join performance optimization

In [1]:
# ======= SETUP =============
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Module04_Joins")
    .config("spark.sql.shuffle.partitions","8")
    .config("spark.driver.memory","4g")
    .config("spark.sql.repl.eagerEval.enabled","true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# LOAD DATA

customers_df = spark.read.csv("data/customers.csv",header=True,inferSchema=True)
orders_df = spark.read.csv("data/orders.csv",header=True,inferSchema=True)
products_df = spark.read.csv("data/products.csv",header=True,inferSchema=True)
order_items_df = spark.read.csv("data/order_items.csv",header=True,inferSchema=True)



In [2]:
# INNER JOIN

# ONLY matching rows from BOTH tables

inner_join = customers_df.join(orders_df,"customer_id","inner")

print(f"Customer : {customers_df.count()}")
print(f"Orders: {orders_df.count()}")
print(f"Inner join: {inner_join.count()}")
print("\ninner Join sample:")
inner_join.select("customer_id","first_name","last_name","order_id","total_amount").show(5)

Customer : 5000
Orders: 50000
Inner join: 50000

inner Join sample:
+-----------+----------+---------+--------+------------+
|customer_id|first_name|last_name|order_id|total_amount|
+-----------+----------+---------+--------+------------+
|       4722|     James|  Jackson|       1|      1784.2|
|       3172|    Daniel|  Jackson|       2|     1536.51|
|       1381|   Anthony|Rodriguez|       3|     1191.88|
|       2332|     David|    Lewis|       4|     1157.78|
|       1330|   Charles|  Sanchez|       5|      1381.5|
+-----------+----------+---------+--------+------------+
only showing top 5 rows


In [ ]:
# Left Join

left_join = customers_df.join(orders_df,"customer_id","left")

print(f"Left join : {left_join.count()}")
print("\nLeft join - customer without orders:")
left_join.filter(
    col("order_id").isNull()).select("customer_id","first_name","last_name").show(5)


Left join : 50000

Left join - customer without orders:
+-----------+----------+---------+
|customer_id|first_name|last_name|
+-----------+----------+---------+
|       4722|     James|  Jackson|
|       3172|    Daniel|  Jackson|
|       1381|   Anthony|Rodriguez|
|       2332|     David|    Lewis|
|       1330|   Charles|  Sanchez|
+-----------+----------+---------+
only showing top 5 rows


In [7]:
# RIGHT JOIN

right_join = customers_df.join(orders_df,"customer_id","right")

print(f"Right join : {right_join.count()}")
print("\nRight join - orders without customer")

right_join.filter(
    col("first_name").isNotNull()).select("order_id","customer_id").show(5)



Right join : 50000

Right join - orders without customer
+--------+-----------+
|order_id|customer_id|
+--------+-----------+
|       1|       4722|
|       2|       3172|
|       3|       1381|
|       4|       2332|
|       5|       1330|
+--------+-----------+
only showing top 5 rows


In [8]:
# FULL OUTER JOIN

full_join = customers_df.join(orders_df,"customer_id","full")
print(f"FullOuter Join: {full_join.count()}")

FullOuter Join: 50000


In [9]:
# CROSS JOIN (CARTESIAN PRODUCT)


# Every row from left + Every row from right
# Dangerous : Creates n*m rows!

# Lets use small datasets for demo

small_customers = customers_df.limit(5)
small_products = products_df.limit(3)

cross_join = small_customers.crossJoin(small_products)
print(f"Small Customers : {small_customers.count()}")
print(f"Small Products: {small_products.count()}")
print(f"Cross join: {cross_join.count()}")


Small Customers : 5
Small Products: 3
Cross join: 15


In [10]:
# LEFT ANTI JOIN

# Rows from left that have no match in Right
# Very Useful for finding 'missing' data

customers_without_orders = customers_df.join(orders_df,"customer_id","left_anti")

print(f"customers_without_orders : {customers_without_orders.count()}")
print(f"\nFirst 5 customers without orders")
customers_without_orders.select("customer_id","first_name","last_name").show(5)

customers_without_orders : 0

First 5 customers without orders
+-----------+----------+---------+
|customer_id|first_name|last_name|
+-----------+----------+---------+
+-----------+----------+---------+



### Join Syntax Options

In [14]:
# 2.1 Different Join Syntaxes

# Method 1 : Column name (simplest)
join1 = customers_df.join(orders_df,"customer_id","inner")

# Method 2 : Join expression (when column name differ)
join2 = customers_df.join(orders_df,customers_df.customer_id==orders_df.customer_id,"inner")

# Method 3 : List of column (for multi-column joins)
join3 = customers_df.join(orders_df,["customer_id"],"inner")

print("All 3 Methods produce same result")
print(f"Method 1: {join1.count()}")
print(f"Method 2: {join2.count()}")
print(f"Method 3: {join3.count()}")

All 3 Methods produce same result
Method 1: 50000
Method 2: 50000
Method 3: 50000


In [16]:
# 2.2 Multi-column Join

multi_col_join = order_items_df.join(products_df,["product_id"],"inner")

print(f"Multi col join result : {multi_col_join.count()}")
multi_col_join.select("item_id","product_name","quantity","unit_price").show(5)

Multi col join result : 60354
+-------+--------------------+--------+----------+
|item_id|        product_name|quantity|unit_price|
+-------+--------------------+--------+----------+
|      1|   Clock - Variant 1|       1|    141.65|
|      3|Dumbbells - Varia...|       3|    381.21|
|      4|Biography - Varia...|       5|    238.88|
|      6| Monitor - Variant 3|       2|    208.53|
|      7|Technical - Varia...|       5|     139.4|
+-------+--------------------+--------+----------+
only showing top 5 rows


In [18]:
# 2.3 Join with suffix for duplicates Columns

# when both dataframes have same column names
df1 = spark.createDataFrame([(1,"a"),(2,"b")],["id","value"])
df2 = spark.createDataFrame([(1,"x"),(2,"y")],["id","value"])

# join with suffix
result = df1.join(df2,"id","inner")
print(f"Without suffix - ambigous columns")
result.printSchema()

# Better rename before join
result1 = df1.join(df2.withColumnRenamed("value","value2"),"id","inner")
print("\nWith renamed column")
result1.printSchema()

Without suffix - ambigous columns
root
 |-- id: long (nullable = true)
 |-- value: string (nullable = true)
 |-- value: string (nullable = true)


With renamed column
root
 |-- id: long (nullable = true)
 |-- value: string (nullable = true)
 |-- value2: string (nullable = true)



### Join Strategies
 ##### Three main join strategies
   - Broadcast Hash join: Small table broadcasted to all executors
   - Sort-Merge-Join: Both tables sorted and merged (default for large tables)
   - Shuffle Hash Join: One side hashed and probed

In [19]:
# 3.1 Broadcasr Hash Join

# Broadcast small table to all executors
# Best for : small table (<10 MB)

from pyspark.sql.functions import broadcast

# Explicit broadcast
broadcast_join = customers_df.join(
    broadcast(orders_df),"customer_id","inner"
)

print(f"broadcast join completed")
print("\nPlan:")
broadcast_join.explain()

broadcast join completed

Plan:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [customer_id#17, first_name#18, last_name#19, email#20, phone#21, city#22, state#23, signup_date#24, customer_segment#25, is_active#26, order_id#44, order_date#46, delivery_date#47, status#48, total_amount#49, discount_amount#50, tax_amount#51, shipping_cost#52, payment_method#53, shipping_address#54]
   +- BroadcastHashJoin [customer_id#17], [customer_id#45], Inner, BuildRight, false
      :- Filter isnotnull(customer_id#17)
      :  +- FileScan csv [customer_id#17,first_name#18,last_name#19,email#20,phone#21,city#22,state#23,signup_date#24,customer_segment#25,is_active#26] Batched: false, DataFilters: [isnotnull(customer_id#17)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/c:/Users/shubh/Desktop/Progress/Developer/PySpark/data/customers..., PartitionFilters: [], PushedFilters: [IsNotNull(customer_id)], ReadSchema: struct<customer_id:int,first_name:string,last_name:string,ema

In [20]:
# 3.2 Sort-Merge Join (Default for large Tables )

# Both tables are sorted by join key, then merge
# Requires shiffule of both tables

sort_merge_join = orders_df.join(order_items_df,"order_id","inner")
print("Sort-Merge join completed")
print("\nPlan:")
sort_merge_join.explain()

Sort-Merge join completed

Plan:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [order_id#44, customer_id#45, order_date#46, delivery_date#47, status#48, total_amount#49, discount_amount#50, tax_amount#51, shipping_cost#52, payment_method#53, shipping_address#54, item_id#99, product_id#101, quantity#102, unit_price#103, total_price#104, discount_percent#105]
   +- BroadcastHashJoin [order_id#44], [order_id#100], Inner, BuildRight, false
      :- Filter isnotnull(order_id#44)
      :  +- FileScan csv [order_id#44,customer_id#45,order_date#46,delivery_date#47,status#48,total_amount#49,discount_amount#50,tax_amount#51,shipping_cost#52,payment_method#53,shipping_address#54] Batched: false, DataFilters: [isnotnull(order_id#44)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/c:/Users/shubh/Desktop/Progress/Developer/PySpark/data/orders.csv], PartitionFilters: [], PushedFilters: [IsNotNull(order_id)], ReadSchema: struct<order_id:int,customer_id:int,order_date:dat

In [ ]:
# 3.3 Controlling Join Stragey

# Force broadcast join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold","10485760") # 10 MB

# Force sort merge join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold","-1")  # Disable broadcast

# Reset to default
spark.conf.set("spark.sql.autoBroadcastJoinThreshold","10485760")

## 4. Practical Example: Complete E-Commerce Analysis

### Scenario:
Create a complete customer order analysis with:
- Customer details
- Order details
- Product details
- Revenue calculations

In [21]:
# 4.1 Complete E-Commerce analysis

# Step-1 Join Orders with Customers

orders_with_customers = orders_df.join(customers_df.select("customer_id","first_name","last_name","city","state","customer_segment"),
"customer_id","inner")

# Step-2 Join with Orders items 
orders_with_items = (
    orders_with_customers.join(order_items_df.select("order_id","product_id","quantity","total_price"),"order_id","inner")
)

# Step-3 Join with Products
complete_data = (
    orders_with_items.join(products_df.select("product_id","product_name","category","brand"),"product_id","inner")
)

print(f"Complete dataset: {complete_data.count()}")
print("\nSample")
complete_data.select("customer_id","first_name","last_name","order_id","product_name","category","quantity","total_price").show(5)

Complete dataset: 60354

Sample
+-----------+----------+---------+--------+--------------------+-----------+--------+-----------+
|customer_id|first_name|last_name|order_id|        product_name|   category|quantity|total_price|
+-----------+----------+---------+--------+--------------------+-----------+--------+-----------+
|       4722|     James|  Jackson|       1|Headphones - Vari...|Electronics|       2|     987.62|
|       3172|    Daniel|  Jackson|       2|   Chair - Variant 1|       Home|       2|      73.18|
|       2332|     David|    Lewis|       4| Romance - Variant 1|      Books|       3|     830.34|
|       2332|     David|    Lewis|       4|  Laptop - Variant 2|Electronics|       4|    1123.48|
|       2363|  Patricia|Hernandez|       8|Resistance Bands ...|     Sports|       2|     819.96|
+-----------+----------+---------+--------+--------------------+-----------+--------+-----------+
only showing top 5 rows


In [22]:
# 4.2 Customer Revenue analysis

customer_revenue = complete_data.groupBy(
    "customer_id","first_name","last_name","customer_segment"
).agg(
    count("order_id").alias("total_orders"),
    sum("quantity").alias("total_items"),
    sum("total_price").alias("total_revenue"),
    count_distinct("product_id").alias("unique_products")
).orderBy(desc("total_revenue"))

print("Top 10 customers by revenue")
customer_revenue.show(10)

Top 10 customers by revenue
+-----------+-----------+---------+----------------+------------+-----------+------------------+---------------+
|customer_id| first_name|last_name|customer_segment|total_orders|total_items|     total_revenue|unique_products|
+-----------+-----------+---------+----------------+------------+-----------+------------------+---------------+
|       3136|    Michael|    Brown|             New|          35|        104|31170.399999999998|             34|
|       4161|Christopher|   Harris|         Regular|          29|        103|29789.160000000003|             28|
|       3069|Christopher|    Brown|             VIP|          32|         95|29658.660000000003|             30|
|        771|    Charles|    White|             New|          30|         92|           27936.8|             28|
|       1312|     Joseph|  Sanchez|             New|          32|         95|27857.980000000003|             31|
|       3639|   Margaret|   Harris|         Premium|          38|   

In [23]:
spark.stop()
print("SparkSession stopped")

SparkSession stopped
